# Train baseline and SFT-filtered chatbot models

This notebook currently trains two GPT-2 conditions: a baseline exposed to reports with fictional PII, and supervised fine-tuning (SFT) after removing PII-containing report examples. Both retain the same general 311 Q&A. Resident prompts are context only; loss is on assistant replies.

DPO preference data is generated, but DPO and GRPO training code is not yet implemented. See the README for the full methods and evaluation plan. The old training checkpoints and losses used a different objective; these runs save to new folders.

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "data" / "synthetic" / "sft_filtered_training_conversations.jsonl").exists(), (
    "Run notebook 01 or python src/generate_data.py first."
)
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage


In [ ]:
# Uncomment in Colab or a fresh environment if dependencies are missing.
# %pip install -r requirements.txt

## Configure and run

`TRAIN_EXAMPLES` limits the number of report examples per condition; general Q&A examples are always retained. The PII-filtered condition may have fewer available reports because PII examples are removed. Set it to `None` for each condition's complete training set. Loss metrics append to `results/training_metrics.csv`, one row per model per epoch.

In [11]:
import subprocess
from datetime import datetime

TRAIN_EXAMPLES = 200  # Report examples per condition; set None for all available.
EPOCHS = 2
RUN_LABEL = f"n{TRAIN_EXAMPLES or 'all'}-e{EPOCHS}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
command = [
    sys.executable,
    str(PROJECT_ROOT / "src" / "train_models.py"),
    "--data-dir", str(PROJECT_ROOT / "data" / "synthetic"),
    "--model-name", "gpt2",
    "--output-dir", str(PROJECT_ROOT / "models" / "four-approach-study" / RUN_LABEL),
    "--metrics-csv", str(PROJECT_ROOT / "results" / "training_metrics.csv"),
    "--seed", "42",
    "--epochs", str(EPOCHS),
    "--batch-size", "1",
    "--gradient-accumulation-steps", "8",
    "--max-length", "128",
]
if TRAIN_EXAMPLES is not None:
    command.extend(["--max-train-examples", str(TRAIN_EXAMPLES)])
print(f"Training baseline and SFT-filtered models: report limit={TRAIN_EXAMPLES}, epochs={EPOCHS}")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

Training baseline and SFT-filtered models: report limit=200, epochs=2

Training baseline with 206 conversations.
Device: mps; base model: gpt2


W1105 18:10:38.160000 68444 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5022.35it/s]
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


baseline: epoch 1/2, answer-only training loss 0.0854, validation loss 0.4974
baseline: epoch 2/2, answer-only training loss 0.0596, validation loss 0.3953


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


Saved baseline model and run configuration to /Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/models/four-approach-study/n200-e2-20261105-181033/baseline

Training sft_filtered with 206 conversations.
Device: mps; base model: gpt2


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5066.45it/s]


sft_filtered: epoch 1/2, answer-only training loss 0.1166, validation loss 0.4616
sft_filtered: epoch 2/2, answer-only training loss 0.0975, validation loss 0.4064


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


Saved sft_filtered model and run configuration to /Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/models/four-approach-study/n200-e2-20261105-181033/sft_filtered


CompletedProcess(args=['/opt/miniconda3/bin/python', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/src/train_models.py', '--data-dir', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/data/synthetic', '--model-name', 'gpt2', '--output-dir', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/models/four-approach-study/n200-e2-20261105-181033', '--metrics-csv', '/Users/eshagupta/Downloads/CMU/Research/pii-chatbot-leakage/results/training_metrics.csv', '--seed', '42', '--epochs', '2', '--batch-size', '1', '--gradient-accumulation-steps', '8', '--max-length', '128', '--max-train-examples', '200'], returncode=0)